# Data Cleaning

In [396]:
import pandas as pd
import numpy as np

In [397]:
df=pd.read_excel("raw.xlsx")

In [398]:
df.head()

,Shipment_ID,Shipment_Status,Shipment_Type,Shipment_Date,Shipment_Volume,Shipment_Event_ID,Ordinal_Shipment_ID,Return_Shipment_ID,Return_To_Sender_Flag,Tracking_ID,...,Payment_Method,Remittance_Amount,Remittance_Timestamp,Reconciliation_Status,Reconciliation_Date,Freight_Charge,Surcharge_Amount,Claim_Amount,Currency_Code,Reporting_Date
0,SHP-100001,created,FORWARD,2026/07/12,4590,SEV-000001,1,NaN,0,trk-9001,...,Prepaid,NaN,NaN,NaN,NaN,₹527.20,₹40.00,NaN,INR,12 July 2026
1,SHP-100001,Picked Up,Forward,12 July 2026,4590,SEV-000002,1,NaN,No,trk-9001,...,Prepaid Online,NaN,NaN,NaN,NaN,527.2,40.00,NaN,INR,2026-07-12
2,SHP-100001,In Transit,Forward,2026/07/12,4590,SEV-000003,1,NaN,0,trk-9001,...,Prepaid Online,NaN,NaN,NaN,NaN,527.2,40.00 INR,NaN,INR,13 July 2026
3,SHP-100001,out_for_delivery,Forward,2026-07-12,4590,SEV-000004,1,NaN,FALSE,trk-9001,...,PREPAID,NaN,NaN,NaN,NaN,527.2,40.00,NaN,₹,16-Jul-2026
4,SHP-100001,Delivered,Forward,2026-07-12,4590,SEV-000005,1,NaN,N,trk-9001,...,PREPAID,NaN,NaN,NaN,NaN,527.20 INR,Rs. 40.00,NaN,inr,2026/07/16


In [399]:
df.tail()

,Shipment_ID,Shipment_Status,Shipment_Type,Shipment_Date,Shipment_Volume,Shipment_Event_ID,Ordinal_Shipment_ID,Return_Shipment_ID,Return_To_Sender_Flag,Tracking_ID,...,Payment_Method,Remittance_Amount,Remittance_Timestamp,Reconciliation_Status,Reconciliation_Date,Freight_Charge,Surcharge_Amount,Claim_Amount,Currency_Code,Reporting_Date
495,SHP-100018,In Transit,Forward,11 August 2026,4620,SEV-000496,1,NaN,no,trk-9016,...,COD,NaN,NaN,NaN,NaN,843.28 INR,NaN,NaN,INR,12 August 2026
496,SHP-100019,In Transit,Forward,2026-09-19,37584,SEV-000497,1,NaN,0,TRK-9017,...,COD,NaN,NaN,NaN,NaN,190.58 INR,25.00,NaN,INR,2026-09-20
497,SHP-100020,In Transit,Forward,04-Jul-2026,11368,SEV-000498,1,NaN,FALSE,trk-9018,...,Cash on Delivery,NaN,NaN,NaN,NaN,779.32,Rs. 25.00,NaN,INR,2026-07-05
498,SHP-100021,In Transit,Forward,2026-07-27,22950,SEV-000499,1,NaN,N,TRK-9019,...,prepaid,NaN,NaN,NaN,NaN,644.66,₹0.00,NaN,INR,2026/07/28
499,SHP-100022,In Transit,Forward,2026-07-05,32760,SEV-000500,1,NaN,N,trk-9020,...,PREPAID,NaN,NaN,NaN,NaN,585.53 INR,NaN,NaN,INR,2026-07-06


# 1. Trim and normalize shipment, parcel, route, stop, and tracking IDs; map carrier aliases to canonical identifiers.

Clean the ID columns

In [400]:
columns = [
    'Shipment_ID',
    'Parcel_ID',
    'Route_ID',
    'Stop_ID',
    'Tracking_ID',
    'Carrier_ID'
]

for col in columns:
    df[col] = df[col].str.strip().str.upper()

Clean carrier names and aliases

In [401]:
df['Carrier_Name'] = df['Carrier_Name'].str.strip().str.upper()

df['Carrier_Alias'] = df['Carrier_Alias'].str.strip().str.upper()

Replace carrier aliases with standard names

In [402]:
df['Carrier_Alias'] = df['Carrier_Alias'].replace({
    'FAST SHIP': 'FASTSHIP',
    'FAST-SHIP': 'FASTSHIP',
    'ROAD X': 'ROADX',
    'ROAD-X': 'ROADX',
    'SWIFT POST': 'SWIFTPOST',
    'SWIFT-POST': 'SWIFTPOST'
})

# 2. Parse event timestamps and normalize timezone offsets to UTC while retaining source timezone when useful.

 Convert timestamp to datetime in UTC

In [403]:
df['Event_Timestamp'] = pd.to_datetime(df['Event_Timestamp'], utc=True,errors='coerce')

-> it is use to find the unique value 

In [404]:
df['Event_Timestamp'].unique()

<DatetimeArray>
['2026-07-12 02:02:00+00:00', '2026-07-12 03:30:00+00:00',
 '2026-07-12 21:15:00+00:00',                       'NaT',
 '2026-07-11 01:39:00+00:00', '2026-07-03 02:42:00+00:00',
 '2026-07-03 04:36:00+00:00', '2026-07-05 01:57:00+00:00',
 '2026-08-13 00:53:00+00:00', '2026-07-13 23:30:00+00:00',
 ...
 '2026-08-01 10:06:00+00:00', '2026-08-29 03:21:00+00:00',
 '2026-08-29 03:57:00+00:00', '2026-08-30 01:02:00+00:00',
 '2026-08-17 02:56:00+00:00', '2026-08-17 19:40:00+00:00',
 '2026-08-20 02:42:00+00:00', '2026-08-20 12:12:00+00:00',
 '2026-07-17 01:13:00+00:00', '2026-09-04 04:30:00+00:00']
Length: 193, dtype: datetime64[ns, UTC]

In [405]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Columns: 154 entries, Shipment_ID to Reporting_Date
dtypes: datetime64[ns, UTC](1), float64(35), int64(8), object(110)
memory usage: 601.7+ KB


# 3. Standardize carrier, service level, warehouse, depot, and delivery-zone names using reference tables.

In [406]:
df['Carrier_Name'] = df['Carrier_Name'].str.strip().str.upper()
df['Service_Level'] = df['Service_Level'].str.strip().str.upper()
df['Warehouse_Name'] = df['Warehouse_Name'].str.strip().str.upper()
df['Depot_Name'] = df['Depot_Name'].str.strip().str.upper()
df['Delivery_Zone'] = df['Delivery_Zone'].str.strip().str.upper()

Replace different names with standard names

In [407]:
df['Carrier_Name'] = df['Carrier_Name'].replace({
    'FAST SHIP': 'FASTSHIP',
    'FAST-SHIP': 'FASTSHIP'})

df['Service_Level'] = df['Service_Level'].replace({
    'SAME DAY': 'SAME-DAY',
    'NEXT DAY': 'NEXT-DAY'
})

In [408]:
df['Carrier_Name'].unique()

array(['FASTSHIP', 'ROADX', 'SWIFTPOST', 'ROAD X', 'ROAD-X', 'GREENLINE',
       'GREEN-LINE', 'SWIFT POST', 'GREEN LINE', 'SWIFT-POST'],
      dtype=object)

In [409]:
df.head()

,Shipment_ID,Shipment_Status,Shipment_Type,Shipment_Date,Shipment_Volume,Shipment_Event_ID,Ordinal_Shipment_ID,Return_Shipment_ID,Return_To_Sender_Flag,Tracking_ID,...,Payment_Method,Remittance_Amount,Remittance_Timestamp,Reconciliation_Status,Reconciliation_Date,Freight_Charge,Surcharge_Amount,Claim_Amount,Currency_Code,Reporting_Date
0,SHP-100001,created,FORWARD,2026/07/12,4590,SEV-000001,1,NaN,0,TRK-9001,...,Prepaid,NaN,NaN,NaN,NaN,₹527.20,₹40.00,NaN,INR,12 July 2026
1,SHP-100001,Picked Up,Forward,12 July 2026,4590,SEV-000002,1,NaN,No,TRK-9001,...,Prepaid Online,NaN,NaN,NaN,NaN,527.2,40.00,NaN,INR,2026-07-12
2,SHP-100001,In Transit,Forward,2026/07/12,4590,SEV-000003,1,NaN,0,TRK-9001,...,Prepaid Online,NaN,NaN,NaN,NaN,527.2,40.00 INR,NaN,INR,13 July 2026
3,SHP-100001,out_for_delivery,Forward,2026-07-12,4590,SEV-000004,1,NaN,FALSE,TRK-9001,...,PREPAID,NaN,NaN,NaN,NaN,527.2,40.00,NaN,₹,16-Jul-2026
4,SHP-100001,Delivered,Forward,2026-07-12,4590,SEV-000005,1,NaN,N,TRK-9001,...,PREPAID,NaN,NaN,NaN,NaN,527.20 INR,Rs. 40.00,NaN,inr,2026/07/16


In [410]:
df.tail()

,Shipment_ID,Shipment_Status,Shipment_Type,Shipment_Date,Shipment_Volume,Shipment_Event_ID,Ordinal_Shipment_ID,Return_Shipment_ID,Return_To_Sender_Flag,Tracking_ID,...,Payment_Method,Remittance_Amount,Remittance_Timestamp,Reconciliation_Status,Reconciliation_Date,Freight_Charge,Surcharge_Amount,Claim_Amount,Currency_Code,Reporting_Date
495,SHP-100018,In Transit,Forward,11 August 2026,4620,SEV-000496,1,NaN,no,TRK-9016,...,COD,NaN,NaN,NaN,NaN,843.28 INR,NaN,NaN,INR,12 August 2026
496,SHP-100019,In Transit,Forward,2026-09-19,37584,SEV-000497,1,NaN,0,TRK-9017,...,COD,NaN,NaN,NaN,NaN,190.58 INR,25.00,NaN,INR,2026-09-20
497,SHP-100020,In Transit,Forward,04-Jul-2026,11368,SEV-000498,1,NaN,FALSE,TRK-9018,...,Cash on Delivery,NaN,NaN,NaN,NaN,779.32,Rs. 25.00,NaN,INR,2026-07-05
498,SHP-100021,In Transit,Forward,2026-07-27,22950,SEV-000499,1,NaN,N,TRK-9019,...,prepaid,NaN,NaN,NaN,NaN,644.66,₹0.00,NaN,INR,2026/07/28
499,SHP-100022,In Transit,Forward,2026-07-05,32760,SEV-000500,1,NaN,N,TRK-9020,...,PREPAID,NaN,NaN,NaN,NaN,585.53 INR,NaN,NaN,INR,2026-07-06


# 4. Normalize shipment statuses (created, picked up, in transit, out for delivery, delivered, returned, exception).

 Convert shipment statuses to lowercase and remove extra spaces

In [411]:
df['Shipment_Status'] = df['Shipment_Status'].str.strip().str.title()


In [412]:
df['Shipment_Status'] = df['Shipment_Status'].replace({
    'pickup': 'picked up',
    'picked-up': 'picked up',
    'return': 'returned',
    'delivery exception': 'exception'
})

In [413]:
df['Shipment_Status'].unique()

array(['Created', 'Picked Up', 'In Transit', 'Out_For_Delivery',
       'Delivered', 'Out For Delivery', 'Picked_Up', 'In_Transit'],
      dtype=object)

In [414]:
print(df['Shipment_Status'].value_counts())

Shipment_Status
Created             101
Delivered            93
Picked Up            80
In Transit           74
Out For Delivery     71
In_Transit           32
Picked_Up            27
Out_For_Delivery     22
Name: count, dtype: int64


In [415]:
df.tail()

,Shipment_ID,Shipment_Status,Shipment_Type,Shipment_Date,Shipment_Volume,Shipment_Event_ID,Ordinal_Shipment_ID,Return_Shipment_ID,Return_To_Sender_Flag,Tracking_ID,...,Payment_Method,Remittance_Amount,Remittance_Timestamp,Reconciliation_Status,Reconciliation_Date,Freight_Charge,Surcharge_Amount,Claim_Amount,Currency_Code,Reporting_Date
495,SHP-100018,In Transit,Forward,11 August 2026,4620,SEV-000496,1,NaN,no,TRK-9016,...,COD,NaN,NaN,NaN,NaN,843.28 INR,NaN,NaN,INR,12 August 2026
496,SHP-100019,In Transit,Forward,2026-09-19,37584,SEV-000497,1,NaN,0,TRK-9017,...,COD,NaN,NaN,NaN,NaN,190.58 INR,25.00,NaN,INR,2026-09-20
497,SHP-100020,In Transit,Forward,04-Jul-2026,11368,SEV-000498,1,NaN,FALSE,TRK-9018,...,Cash on Delivery,NaN,NaN,NaN,NaN,779.32,Rs. 25.00,NaN,INR,2026-07-05
498,SHP-100021,In Transit,Forward,2026-07-27,22950,SEV-000499,1,NaN,N,TRK-9019,...,prepaid,NaN,NaN,NaN,NaN,644.66,₹0.00,NaN,INR,2026/07/28
499,SHP-100022,In Transit,Forward,2026-07-05,32760,SEV-000500,1,NaN,N,TRK-9020,...,PREPAID,NaN,NaN,NaN,NaN,585.53 INR,NaN,NaN,INR,2026-07-06


# 5. Deduplicate repeated scan events using tracking ID, event code, timestamp, location, and source priority.

In [416]:
df.columns = df.columns.str.strip()
print(df.columns.tolist())


['Shipment_ID', 'Shipment_Status', 'Shipment_Type', 'Shipment_Date', 'Shipment_Volume', 'Shipment_Event_ID', 'Ordinal_Shipment_ID', 'Return_Shipment_ID', 'Return_To_Sender_Flag', 'Tracking_ID', 'Parcel_ID', 'Parcel_Count', 'Parcel_Weight', 'Parcel_Volume', 'Parcel_Status', 'Package_Weight', 'Weight', 'Weight_Unit', 'Length', 'Width', 'Height', 'Dimension_Unit', 'Declared_Value', 'Carrier_ID', 'Carrier_Name', 'Carrier_Alias', 'Service_Type', 'Service_Level', 'Booking_Date', 'Booking_Weekday', 'Pickup_Timestamp', 'Actual_Delivery_Timestamp', 'Delivery_Timestamp', 'Promised_Delivery_Date', 'Promised_Delivery_Timestamp', 'Promised_Pickup_Timestamp', 'Promised_Delivery_Start', 'Promised_Delivery_End', 'Delivery_Attempt_Count', 'Delivery_Attempt_ID', 'Delivery_Attempt_Number', 'First_Attempt_Timestamp', 'First_Attempt_Delivery_Flag', 'First_Attempt_Status', 'Attempt_Number', 'Attempt_Status', 'Attempt_Timestamp', 'Delivery_Success_Flag', 'On_Time_Delivery_Flag', 'Delivery_Location', 'Deliver

basically in our data we dont have the column name location so we use (Depot_name) insted of location


In [417]:
df = df.drop_duplicates(
    subset=['Tracking_ID', 'Event_Code', 'Event_Timestamp','Depot_Name'],
    keep='first'
)

In [418]:
df['Depot_Name'].unique()

array([nan, 'BHUBANESWAR DEPOT', 'CUTTACK DEPOT', 'BERHAMPUR DEPOT',
       'SAMBALPUR DEPOT', 'PURI DEPOT', 'ROURKELA DEPOT'], dtype=object)

In [419]:
print(df.columns.tolist())

['Shipment_ID', 'Shipment_Status', 'Shipment_Type', 'Shipment_Date', 'Shipment_Volume', 'Shipment_Event_ID', 'Ordinal_Shipment_ID', 'Return_Shipment_ID', 'Return_To_Sender_Flag', 'Tracking_ID', 'Parcel_ID', 'Parcel_Count', 'Parcel_Weight', 'Parcel_Volume', 'Parcel_Status', 'Package_Weight', 'Weight', 'Weight_Unit', 'Length', 'Width', 'Height', 'Dimension_Unit', 'Declared_Value', 'Carrier_ID', 'Carrier_Name', 'Carrier_Alias', 'Service_Type', 'Service_Level', 'Booking_Date', 'Booking_Weekday', 'Pickup_Timestamp', 'Actual_Delivery_Timestamp', 'Delivery_Timestamp', 'Promised_Delivery_Date', 'Promised_Delivery_Timestamp', 'Promised_Pickup_Timestamp', 'Promised_Delivery_Start', 'Promised_Delivery_End', 'Delivery_Attempt_Count', 'Delivery_Attempt_ID', 'Delivery_Attempt_Number', 'First_Attempt_Timestamp', 'First_Attempt_Delivery_Flag', 'First_Attempt_Status', 'Attempt_Number', 'Attempt_Status', 'Attempt_Timestamp', 'Delivery_Success_Flag', 'On_Time_Delivery_Flag', 'Delivery_Location', 'Deliver

# 6. Normalize weight and dimension units (g/kg, cm/m, lb/in) with explicit conversion rules.

Convert weight to kilograms

In [420]:
df['Weight_kg'] = df['Weight'] * 1.0

df.loc[df['Weight_Unit'] == 'g', 'Weight_kg'] = df['Weight'] / 1000
df.loc[df['Weight_Unit'] == 'lb', 'Weight_kg'] = df['Weight'] * 0.453592


Convert length to centimeters

In [421]:
df['Length'] = df['Length'].replace('?', pd.NA)

In [422]:
df['Length'] = pd.to_numeric(df['Length'], errors='coerce')

In [423]:
df['Length_cm'] = df['Length']

df.loc[df['Dimension_Unit'] == 'm', 'Length_cm'] = df['Length'] * 100

df.loc[df['Dimension_Unit'] == 'in', 'Length_cm'] = df['Length'] * 2.54

In [424]:
df['Dimension_Unit'].unique()

array(['"', 'IN', 'inch', 'inches', 'in', 'Cm', 'centimeter', 'CM', 'cm',
       'M', 'm', 'meter', 'mtr'], dtype=object)

In [425]:
print(df['Length'].head(10))
print(df['Length'].dtype)

0      NaN
1    11.81
2    11.81
3    11.81
4    11.81
5    11.81
6    20.47
7    20.47
8    20.47
9    20.47
Name: Length, dtype: float64
float64


# 7. Validate package weight, dimensions, declared value, and parcel count against allowable ranges.

Check invalid weight

In [426]:
df.loc[df['Weight'] <= 0, 'Weight'] = None

Check invalid dimensions

In [427]:
df['Length'] = pd.to_numeric(df['Length'], errors='coerce')
df['Width'] = pd.to_numeric(df['Width'], errors='coerce')
df['Height'] = pd.to_numeric(df['Height'], errors='coerce')

df.loc[df['Length'] <= 0, 'Length'] = None
df.loc[df['Width'] <= 0, 'Width'] = None
df.loc[df['Height'] <= 0, 'Height'] = None

 Check invalid declared value

In [428]:
df['Declared_Value'] = pd.to_numeric(df['Declared_Value'], errors='coerce')
df.loc[df['Declared_Value'] < 0, 'Declared_Value'] = None

Check invalid parcel count

In [429]:
df['Parcel_Count'] = pd.to_numeric(df['Parcel_Count'], errors='coerce')
df.loc[df['Parcel_Count'] <= 0, 'Parcel_Count'] = None


In [430]:
df['Declared_Value'].unique()

array([   nan, 11387.,  6843.,  4862.,   452.,  5411.,  3239., 12334.,
        5746.,  3563., 10386.,  5853.,  5042.,  8612., 10650.,  3579.,
        5173.,  4220., 14965.,  6798.,  5214.,  6653.,  3025.,  3299.,
       11792.,  2546.,  7728.,  1129.,  1275.,  3412.,  9583.,  4708.,
        8980.,  5233.,  8214., 12043.,  6649.,  1939.,  6471.,  1900.,
        6942.,  1187., 14208.,  2975.,  5844., 13971., 12549.,  6186.,
       12887., 12246., 12300.,  8087.,  8583., 10120., 10632.,  3084.,
       12639.,  6409.,  9374., 13686.,  9375.,  9675., 14434., 11485.,
       10566., 10342.,  3426.,   597.,  8242.,  5869., 12179.,  1446.,
        5659.,  5614.,  8012., 13640.,   572., 14265., 10995., 10945.,
        1556.,   395.,  3773.,  5081.])

In [431]:
df.head()

,Shipment_ID,Shipment_Status,Shipment_Type,Shipment_Date,Shipment_Volume,Shipment_Event_ID,Ordinal_Shipment_ID,Return_Shipment_ID,Return_To_Sender_Flag,Tracking_ID,...,Remittance_Timestamp,Reconciliation_Status,Reconciliation_Date,Freight_Charge,Surcharge_Amount,Claim_Amount,Currency_Code,Reporting_Date,Weight_kg,Length_cm
0,SHP-100001,Created,FORWARD,2026/07/12,4590,SEV-000001,1,NaN,0,TRK-9001,...,NaN,NaN,NaN,₹527.20,₹40.00,NaN,INR,12 July 2026,1500.0,NaN
1,SHP-100001,Picked Up,Forward,12 July 2026,4590,SEV-000002,1,NaN,No,TRK-9001,...,NaN,NaN,NaN,527.2,40.00,NaN,INR,2026-07-12,1500.0,11.81
2,SHP-100001,In Transit,Forward,2026/07/12,4590,SEV-000003,1,NaN,0,TRK-9001,...,NaN,NaN,NaN,527.2,40.00 INR,NaN,INR,13 July 2026,1500.0,11.81
3,SHP-100001,Out_For_Delivery,Forward,2026-07-12,4590,SEV-000004,1,NaN,FALSE,TRK-9001,...,NaN,NaN,NaN,527.2,40.00,NaN,₹,16-Jul-2026,1500.0,11.81
4,SHP-100001,Delivered,Forward,2026-07-12,4590,SEV-000005,1,NaN,N,TRK-9001,...,NaN,NaN,NaN,527.20 INR,Rs. 40.00,NaN,inr,2026/07/16,1500.0,11.81


# 8. Standardize postal codes, country codes, and depot/location identifiers.

In [432]:

df['Postal_Code'] = df['Postal_Code'].astype('string').str.strip().str.upper().str.replace(' ', '').str.replace('-', '')

df['Country_Code'] = df['Country_Code'].astype('string').str.strip().str.upper().replace({
    'INDIAD': 'IND',
    'INDIADIA': 'IND',
    'INDIA': 'IND'
})

df['Depot_ID'] = df['Depot_ID'].astype('string').str.strip().str.upper()

df['Location_ID'] = df['Location_ID'].astype('string').str.strip().str.upper()


In [433]:
df['Country_Code'].unique()

<StringArray>
['IND', 'IN']
Length: 2, dtype: string

In [434]:
df['Location_ID'].unique()

<StringArray>
['LOC-NORTH', 'LOC-HUB01',  'LOC-BBSR', 'LOC-SOUTH',   'LOC-CTC',   'LOC-BER',
  'LOC-EAST',  'LOC-WEST',   'LOC-SBP',  'LOC-PURI',   'LOC-RKL']
Length: 11, dtype: string

In [435]:
df['Postal_Code'].unique()

<StringArray>
['751001', '753001', '760001', '768001', '752001', '769001']
Length: 6, dtype: string

In [436]:
df.head()

,Shipment_ID,Shipment_Status,Shipment_Type,Shipment_Date,Shipment_Volume,Shipment_Event_ID,Ordinal_Shipment_ID,Return_Shipment_ID,Return_To_Sender_Flag,Tracking_ID,...,Remittance_Timestamp,Reconciliation_Status,Reconciliation_Date,Freight_Charge,Surcharge_Amount,Claim_Amount,Currency_Code,Reporting_Date,Weight_kg,Length_cm
0,SHP-100001,Created,FORWARD,2026/07/12,4590,SEV-000001,1,NaN,0,TRK-9001,...,NaN,NaN,NaN,₹527.20,₹40.00,NaN,INR,12 July 2026,1500.0,NaN
1,SHP-100001,Picked Up,Forward,12 July 2026,4590,SEV-000002,1,NaN,No,TRK-9001,...,NaN,NaN,NaN,527.2,40.00,NaN,INR,2026-07-12,1500.0,11.81
2,SHP-100001,In Transit,Forward,2026/07/12,4590,SEV-000003,1,NaN,0,TRK-9001,...,NaN,NaN,NaN,527.2,40.00 INR,NaN,INR,13 July 2026,1500.0,11.81
3,SHP-100001,Out_For_Delivery,Forward,2026-07-12,4590,SEV-000004,1,NaN,FALSE,TRK-9001,...,NaN,NaN,NaN,527.2,40.00,NaN,₹,16-Jul-2026,1500.0,11.81
4,SHP-100001,Delivered,Forward,2026-07-12,4590,SEV-000005,1,NaN,N,TRK-9001,...,NaN,NaN,NaN,527.20 INR,Rs. 40.00,NaN,inr,2026/07/16,1500.0,11.81


In [437]:
df.tail()

,Shipment_ID,Shipment_Status,Shipment_Type,Shipment_Date,Shipment_Volume,Shipment_Event_ID,Ordinal_Shipment_ID,Return_Shipment_ID,Return_To_Sender_Flag,Tracking_ID,...,Remittance_Timestamp,Reconciliation_Status,Reconciliation_Date,Freight_Charge,Surcharge_Amount,Claim_Amount,Currency_Code,Reporting_Date,Weight_kg,Length_cm
490,SHP-100007,Picked Up,Forward,2026-07-18,21168,SEV-000491,1,NaN,n,TRK-9007,...,NaN,NaN,NaN,399.37,₹40.00,NaN,inr,"Jul 18, 2026",-1.81,19.29
491,SHP-100008,Picked Up,Forward,2026/07/28,10166,SEV-000492,1,NaN,N,TRK-9008,...,NaN,NaN,NaN,824.58,₹25.00,NaN,Rs,2026-07-28,17.36,34.00
492,SHP-100009,Picked_Up,Forward,2026-08-10,6156,SEV-000493,1,NaN,FALSE,TRK-9009,...,NaN,NaN,NaN,Rs. 660.58,0.00,NaN,INR,"Aug 10, 2026",5.19,27.00
493,SHP-100010,Picked_Up,Forward,06-Aug-2026,3480,SEV-000494,1,NaN,No,TRK-9010,...,NaN,NaN,NaN,Rs. 285.30,NaN,NaN,inr,2026-08-06,16.99,0.20
494,SHP-100011,Picked Up,Forward,2026/09/03,23220,SEV-000495,1,NaN,N,TRK-9011,...,NaN,NaN,NaN,525.61,Rs. 25.00,NaN,Rs,"Sep 03, 2026",10.56,43.00


# 9. Validate GPS latitude/longitude bounds and flag implausible jumps or missing coordinates.

Check invalid latitude and longitude

In [438]:
 
df.loc[(df['Latitude'] < -90) | (df['Latitude'] > 90), 'Latitude'] = None

df.loc[(df['Longitude'] < -180) | (df['Longitude'] > 180), 'Longitude'] = None




 Check GPS jumps between consecutive rows

In [439]:
df['GPS_Jump'] = False

df.loc[df['Latitude'].diff().abs() > 1, 'GPS_Jump'] = True
df.loc[df['Longitude'].diff().abs() > 1, 'GPS_Jump'] = True

Check missing coordinates

In [440]:
df[df['Latitude'].isna() | df['Longitude'].isna()]

,Shipment_ID,Shipment_Status,Shipment_Type,Shipment_Date,Shipment_Volume,Shipment_Event_ID,Ordinal_Shipment_ID,Return_Shipment_ID,Return_To_Sender_Flag,Tracking_ID,...,Reconciliation_Status,Reconciliation_Date,Freight_Charge,Surcharge_Amount,Claim_Amount,Currency_Code,Reporting_Date,Weight_kg,Length_cm,GPS_Jump
32,SHP-100008,In_Transit,Forward,2026/07/28,10166,SEV-000033,1,NaN,0,TRK-9008,...,NaN,NaN,Rs. 824.58,INR 25,NaN,INR,"Jul 29, 2026",17.360000,34.0,False
37,SHP-100009,In_Transit,Forward,10-Aug-2026,6156,SEV-000038,1,NaN,FALSE,TRK-9009,...,NaN,NaN,660.58,0,NaN,INR,2026-08-11,5190.000000,27.0,False
42,SHP-100010,In Transit,Forward,2026-08-06,3480,SEV-000043,1,NaN,FALSE,TRK-9010,...,NaN,NaN,285.30,0,NaN,INR,"Aug 07, 2026",16.991556,0.2,False
47,SHP-100011,In_Transit,Forward,2026/09/03,23220,SEV-000048,1,NaN,n,TRK-9011,...,NaN,NaN,525.61,₹25.00,NaN,INR,2026/09/04,23.280000,43.0,True


In [441]:
df['Latitude'].unique()

array([20.95566, 20.95089, 20.66055, 20.3015 , 20.29215, 19.50361,
       19.50944, 19.98011, 20.47292, 20.4717 , 19.49497, 19.49101,
       19.39654, 19.32492, 19.30441, 20.40864, 20.40011, 20.42543,
       20.45483, 20.46733, 19.49357, 19.50799, 20.00323, 20.40437,
       20.39498, 19.88408, 21.10448, 21.09758, 20.80937, 20.3984 ,
       20.3993 ,      nan, 21.46911, 21.46583, 19.49131, 19.50213,
       20.3    , 21.45855, 21.45149, 21.09696, 21.09176, 20.27983,
       20.31531, 20.40931, 20.39939, 20.28275, 20.2829 , 19.50256,
       19.49264, 19.65358, 19.80824, 19.81941, 19.49667, 19.50801,
       19.84963, 20.31392, 20.29571, 20.39707, 20.397  , 21.37235,
       22.25855, 22.25983, 20.94749, 20.94962, 20.59753, 20.39901,
       20.39728, 20.09009, 20.39729, 20.39736, 20.3535 , 19.49095,
       19.49858, 19.6458 , 20.39546, 20.40454, 19.85189, 19.33311,
       19.32216, 21.1097 , 21.09619, 21.70698, 22.27424, 22.24725,
       20.95069, 20.94489, 20.13284, 19.29681, 19.30015, 20.40

In [442]:
df['Longitude'].unique()

array([85.09531, 85.091  , 85.44951, 85.82587, 85.80715, 84.89229,
       84.90524, 85.35269, 85.89363, 85.87105, 84.8938 , 84.90999,
       84.8897 , 84.77639, 84.7789 , 86.19207, 86.19502, 86.06207,
       85.89896, 85.86918, 84.90925, 85.37153, 86.19768, 86.20697,
       85.49918, 84.05681, 84.04423, 84.98168, 86.1998 , 86.20651,
       85.8    , 83.97081, 83.98244, 84.89983, 84.90581,      nan,
       83.97046, 83.97019, 84.04667, 84.04505, 85.84419, 85.81425,
       86.19664, 86.20938, 85.1    , 85.81648, 85.80766, 84.90807,
       84.90079, 85.32313, 85.84116, 85.83662, 84.90978, 84.89006,
       85.33879, 85.80623, 85.84089, 86.20331, 86.20113, 85.5119 ,
       84.8399 , 84.85128, 85.0966 , 85.09471, 85.42614, 86.20622,
       86.20836, 86.02387, 86.20439, 86.20894, 85.97251, 84.8984 ,
       84.9023 , 85.36261, 86.20599, 86.19158, 85.49176, 84.77981,
       84.81205, 84.05584, 84.05999, 84.4147 , 84.84362, 84.85858,
       85.09132, 85.10842, 84.98453, 84.77615, 84.77797, 86.20

# 10. Normalize route and stop sequence numbers; detect duplicate or skipped stops.

Convert stop sequence to numbers

In [443]:
df['Stop_Sequence'] = pd.to_numeric(df['Stop_Sequence'], errors='coerce')

 duplicate stops

In [444]:
df['Duplicate_Stop'] = df.duplicated(
    subset=['Route_ID', 'Stop_Sequence']
)

skipped stops

In [445]:
df = df.sort_values(['Route_ID', 'Stop_Sequence'])

df['Skipped_Stop'] = df.groupby('Route_ID')['Stop_Sequence'].diff() > 1


In [446]:
df[['Route_ID', 'Stop_Sequence', 'Duplicate_Stop', 'Skipped_Stop']].head()

,Route_ID,Stop_Sequence,Duplicate_Stop,Skipped_Stop
315,RT-BBSR-0714,4.0,False,False
316,RT-BBSR-0714,4.0,True,False
430,RT-BBSR-0715,14.0,False,False
431,RT-BBSR-0715,14.0,True,False
260,RT-BBSR-0715,15.0,False,False


# 11. Link proof-of-delivery events to the correct shipment and delivery attempt.

In [447]:
df.columns[df.columns.str.contains('POD|Attempt|Shipment', case=False)]

Index(['Shipment_ID', 'Shipment_Status', 'Shipment_Type', 'Shipment_Date',
       'Shipment_Volume', 'Shipment_Event_ID', 'Ordinal_Shipment_ID',
       'Return_Shipment_ID', 'Delivery_Attempt_Count', 'Delivery_Attempt_ID',
       'Delivery_Attempt_Number', 'First_Attempt_Timestamp',
       'First_Attempt_Delivery_Flag', 'First_Attempt_Status', 'Attempt_Number',
       'Attempt_Status', 'Attempt_Timestamp', 'POD_Status', 'POD_Timestamp'],
      dtype='object')

link and display POD details with each shipment and delivery attempt

In [448]:
df = df.sort_values(['Shipment_ID', 'Delivery_Attempt_ID'])

df[['Shipment_ID',
          'Delivery_Attempt_ID',
          'POD_Status',
          'POD_Timestamp']].head()

,Shipment_ID,Delivery_Attempt_ID,POD_Status,POD_Timestamp
4,SHP-100001,ATT-100001-1,CAPTURED,2026-07-16T16:14:00+05:30
3,SHP-100001,ATT-100001-1,NaN,NaN
0,SHP-100001,NaN,NaN,NaN
1,SHP-100001,NaN,NaN,NaN
2,SHP-100001,NaN,NaN,NaN


In [449]:
df.tail()

,Shipment_ID,Shipment_Status,Shipment_Type,Shipment_Date,Shipment_Volume,Shipment_Event_ID,Ordinal_Shipment_ID,Return_Shipment_ID,Return_To_Sender_Flag,Tracking_ID,...,Freight_Charge,Surcharge_Amount,Claim_Amount,Currency_Code,Reporting_Date,Weight_kg,Length_cm,GPS_Jump,Duplicate_Stop,Skipped_Stop
483,SHP-100100,Picked Up,forward,15-Jul-2026,4290,SEV-000484,1,NaN,0,TRK-9098,...,309.73 INR,₹40.00,NaN,Rs,15 July 2026,16.14,15.0,False,True,False
484,SHP-100100,In Transit,Forward,"Jul 15, 2026",4290,SEV-000485,1,NaN,n,TRK-9098,...,Rs. 309.73,Rs. 40.00,NaN,INR,16-Jul-2026,16.14,15.0,False,True,False
487,SHP-100101,Created,Forward,2026/09/04,7018,SEV-000488,1,NaN,FALSE,TRK-9099,...,INR 201,INR 15,NaN,Rs,"Sep 04, 2026",5.05,29.0,False,True,False
488,SHP-100101,Picked Up,Forward,2026/09/04,7018,SEV-000489,1,NaN,no,TRK-9099,...,201.47,15,NaN,inr,"Sep 04, 2026",5.05,29.0,False,True,False
489,SHP-100101,In Transit,Forward,2026-09-04,7018,SEV-000490,1,NaN,FALSE,TRK-9099,...,201.47 INR,15.00,NaN,INR,"Sep 05, 2026",5.05,29.0,False,True,False


# 12. Normalize exception and failure reason codes (recipient unavailable, address issue, damaged, weather).

Clean exception and failure reason codes

In [450]:

df['Exception_Reason'] = df['Exception_Reason'].str.strip().str.lower()

In [451]:
df['Exception_Reason'] = df['Exception_Reason'].replace({
    'recipient not available': 'recipient unavailable',
    'address problem': 'address issue',
    'parcel damaged': 'damaged',
    'bad weather': 'weather'
})




In [452]:
df['Exception_Reason'].value_counts()

Exception_Reason
heavy rain delay         4
recipient unavailable    1
address incomplete       1
Name: count, dtype: int64

In [453]:
df.tail()

,Shipment_ID,Shipment_Status,Shipment_Type,Shipment_Date,Shipment_Volume,Shipment_Event_ID,Ordinal_Shipment_ID,Return_Shipment_ID,Return_To_Sender_Flag,Tracking_ID,...,Freight_Charge,Surcharge_Amount,Claim_Amount,Currency_Code,Reporting_Date,Weight_kg,Length_cm,GPS_Jump,Duplicate_Stop,Skipped_Stop
483,SHP-100100,Picked Up,forward,15-Jul-2026,4290,SEV-000484,1,NaN,0,TRK-9098,...,309.73 INR,₹40.00,NaN,Rs,15 July 2026,16.14,15.0,False,True,False
484,SHP-100100,In Transit,Forward,"Jul 15, 2026",4290,SEV-000485,1,NaN,n,TRK-9098,...,Rs. 309.73,Rs. 40.00,NaN,INR,16-Jul-2026,16.14,15.0,False,True,False
487,SHP-100101,Created,Forward,2026/09/04,7018,SEV-000488,1,NaN,FALSE,TRK-9099,...,INR 201,INR 15,NaN,Rs,"Sep 04, 2026",5.05,29.0,False,True,False
488,SHP-100101,Picked Up,Forward,2026/09/04,7018,SEV-000489,1,NaN,no,TRK-9099,...,201.47,15,NaN,inr,"Sep 04, 2026",5.05,29.0,False,True,False
489,SHP-100101,In Transit,Forward,2026-09-04,7018,SEV-000490,1,NaN,FALSE,TRK-9099,...,201.47 INR,15.00,NaN,INR,"Sep 05, 2026",5.05,29.0,False,True,False


# 13. Standardize delivery promise dates and distinguish promised date from actual delivery timestamp.

 Convert dates into datetime format

In [454]:

df['Promised_Delivery_Date'] = pd.to_datetime(
    df['Promised_Delivery_Date'], errors='coerce'
)

df['Actual_Delivery_Timestamp'] = pd.to_datetime(
    df['Actual_Delivery_Timestamp'], errors='coerce', utc=True
)

In [455]:
df[['Promised_Delivery_Date', 'Actual_Delivery_Timestamp']].head()

,Promised_Delivery_Date,Actual_Delivery_Timestamp
4,2026-07-16,2026-07-16 10:44:00+00:00
3,2026-07-16,NaT
0,2026-07-16,NaT
1,2026-07-16,NaT
2,2026-07-16,NaT


In [456]:
df.tail()

,Shipment_ID,Shipment_Status,Shipment_Type,Shipment_Date,Shipment_Volume,Shipment_Event_ID,Ordinal_Shipment_ID,Return_Shipment_ID,Return_To_Sender_Flag,Tracking_ID,...,Freight_Charge,Surcharge_Amount,Claim_Amount,Currency_Code,Reporting_Date,Weight_kg,Length_cm,GPS_Jump,Duplicate_Stop,Skipped_Stop
483,SHP-100100,Picked Up,forward,15-Jul-2026,4290,SEV-000484,1,NaN,0,TRK-9098,...,309.73 INR,₹40.00,NaN,Rs,15 July 2026,16.14,15.0,False,True,False
484,SHP-100100,In Transit,Forward,"Jul 15, 2026",4290,SEV-000485,1,NaN,n,TRK-9098,...,Rs. 309.73,Rs. 40.00,NaN,INR,16-Jul-2026,16.14,15.0,False,True,False
487,SHP-100101,Created,Forward,2026/09/04,7018,SEV-000488,1,NaN,FALSE,TRK-9099,...,INR 201,INR 15,NaN,Rs,"Sep 04, 2026",5.05,29.0,False,True,False
488,SHP-100101,Picked Up,Forward,2026/09/04,7018,SEV-000489,1,NaN,no,TRK-9099,...,201.47,15,NaN,inr,"Sep 04, 2026",5.05,29.0,False,True,False
489,SHP-100101,In Transit,Forward,2026-09-04,7018,SEV-000490,1,NaN,FALSE,TRK-9099,...,201.47 INR,15.00,NaN,INR,"Sep 05, 2026",5.05,29.0,False,True,False


# 14. Handle missing pickup or delivery events as unknown rather than assuming they occurred.

In [457]:
df['Attempt_Status'] = df['Attempt_Status'].fillna('Unknown')
df['POD_Status'] = df['POD_Status'].fillna('Unknown')

df[['Attempt_Status', 'POD_Status']].head()

,Attempt_Status,POD_Status
4,SUCCESS,CAPTURED
3,in_progress,Unknown
0,Unknown,Unknown
1,Unknown,Unknown
2,Unknown,Unknown


# 15. Flag impossible event orderings, such as delivered before pickup, and out-of-order late-arriving scans.

Convert timestamps into datetime format

In [458]:

df['First_Attempt_Timestamp'] = pd.to_datetime(df['First_Attempt_Timestamp'], errors='coerce', utc=True)
df['POD_Timestamp'] = pd.to_datetime(df['POD_Timestamp'], errors='coerce', utc=True)


 Flag delivery before pickup/attempt

In [459]:
df['Impossible_Order'] = df['POD_Timestamp'] < df['First_Attempt_Timestamp']

df[['Shipment_ID', 'First_Attempt_Timestamp', 'POD_Timestamp', 'Impossible_Order']].head()

,Shipment_ID,First_Attempt_Timestamp,POD_Timestamp,Impossible_Order
4,SHP-100001,NaT,2026-07-16 10:44:00+00:00,False
3,SHP-100001,NaT,NaT,False
0,SHP-100001,NaT,NaT,False
1,SHP-100001,NaT,NaT,False
2,SHP-100001,NaT,NaT,False


In [460]:
df.tail()

,Shipment_ID,Shipment_Status,Shipment_Type,Shipment_Date,Shipment_Volume,Shipment_Event_ID,Ordinal_Shipment_ID,Return_Shipment_ID,Return_To_Sender_Flag,Tracking_ID,...,Surcharge_Amount,Claim_Amount,Currency_Code,Reporting_Date,Weight_kg,Length_cm,GPS_Jump,Duplicate_Stop,Skipped_Stop,Impossible_Order
483,SHP-100100,Picked Up,forward,15-Jul-2026,4290,SEV-000484,1,NaN,0,TRK-9098,...,₹40.00,NaN,Rs,15 July 2026,16.14,15.0,False,True,False,False
484,SHP-100100,In Transit,Forward,"Jul 15, 2026",4290,SEV-000485,1,NaN,n,TRK-9098,...,Rs. 40.00,NaN,INR,16-Jul-2026,16.14,15.0,False,True,False,False
487,SHP-100101,Created,Forward,2026/09/04,7018,SEV-000488,1,NaN,FALSE,TRK-9099,...,INR 15,NaN,Rs,"Sep 04, 2026",5.05,29.0,False,True,False,False
488,SHP-100101,Picked Up,Forward,2026/09/04,7018,SEV-000489,1,NaN,no,TRK-9099,...,15,NaN,inr,"Sep 04, 2026",5.05,29.0,False,True,False,False
489,SHP-100101,In Transit,Forward,2026-09-04,7018,SEV-000490,1,NaN,FALSE,TRK-9099,...,15.00,NaN,INR,"Sep 05, 2026",5.05,29.0,False,True,False,False


In [461]:
df.head()

,Shipment_ID,Shipment_Status,Shipment_Type,Shipment_Date,Shipment_Volume,Shipment_Event_ID,Ordinal_Shipment_ID,Return_Shipment_ID,Return_To_Sender_Flag,Tracking_ID,...,Surcharge_Amount,Claim_Amount,Currency_Code,Reporting_Date,Weight_kg,Length_cm,GPS_Jump,Duplicate_Stop,Skipped_Stop,Impossible_Order
4,SHP-100001,Delivered,Forward,2026-07-12,4590,SEV-000005,1,NaN,N,TRK-9001,...,Rs. 40.00,NaN,inr,2026/07/16,1500.0,11.81,False,True,False,False
3,SHP-100001,Out_For_Delivery,Forward,2026-07-12,4590,SEV-000004,1,NaN,FALSE,TRK-9001,...,40.00,NaN,₹,16-Jul-2026,1500.0,11.81,False,False,False,False
0,SHP-100001,Created,FORWARD,2026/07/12,4590,SEV-000001,1,NaN,0,TRK-9001,...,₹40.00,NaN,INR,12 July 2026,1500.0,NaN,False,False,False,False
1,SHP-100001,Picked Up,Forward,12 July 2026,4590,SEV-000002,1,NaN,No,TRK-9001,...,40.00,NaN,INR,2026-07-12,1500.0,11.81,False,True,False,False
2,SHP-100001,In Transit,Forward,2026/07/12,4590,SEV-000003,1,NaN,0,TRK-9001,...,40.00 INR,NaN,INR,13 July 2026,1500.0,11.81,False,True,False,False


# 16. Identify test shipments, cancelled labels, and voided consignments using documented exclusion rules.

In [462]:
df['Exclusion_Reason'] = df['Shipment_Status'].str.lower().map({
    'test': 'Test Shipment',
    'cancelled': 'Cancelled Label',
    'canceled': 'Cancelled Label',
    'voided': 'Voided Consignment'
})

df[['Shipment_ID', 'Shipment_Status', 'Exclusion_Reason']].head()

,Shipment_ID,Shipment_Status,Exclusion_Reason
4,SHP-100001,Delivered,NaN
3,SHP-100001,Out_For_Delivery,NaN
0,SHP-100001,Created,NaN
1,SHP-100001,Picked Up,NaN
2,SHP-100001,In Transit,NaN


# 17. Normalize freight charges, surcharges, COD amounts, and claim amounts to numeric values and currencies.

In [463]:
df['Freight_Charge'] = pd.to_numeric(df['Freight_Charge'], errors='coerce')
df['Surcharge_Amount'] = pd.to_numeric(df['Surcharge_Amount'], errors='coerce')
df['COD_Amount'] = pd.to_numeric(df['COD_Amount'], errors='coerce')
df['Claim_Amount'] = pd.to_numeric(df['Claim_Amount'], errors='coerce')

df[['Freight_Charge', 'Surcharge_Amount', 'COD_Amount', 'Claim_Amount']].head()

,Freight_Charge,Surcharge_Amount,COD_Amount,Claim_Amount
4,NaN,NaN,NaN,NaN
3,527.2,40.0,NaN,NaN
0,NaN,NaN,NaN,NaN
1,527.2,40.0,NaN,NaN
2,527.2,NaN,NaN,NaN


# 18. Validate return-to-sender records against original shipment identifiers.

Check return shipments

In [464]:

df['Return_Shipment_ID'] = df['Return_Shipment_ID'].astype('string').str.strip()

df['Invalid_Return'] = (
    (df['Shipment_Status'].str.lower() == 'returned') &
    (df['Return_Shipment_ID'].isna() | (df['Return_Shipment_ID'] == ''))
)


df[['Shipment_ID', 'Shipment_Status', 'Return_Shipment_ID', 'Invalid_Return']].head()

,Shipment_ID,Shipment_Status,Return_Shipment_ID,Invalid_Return
4,SHP-100001,Delivered,<NA>,False
3,SHP-100001,Out_For_Delivery,<NA>,False
0,SHP-100001,Created,<NA>,False
1,SHP-100001,Picked Up,<NA>,False
2,SHP-100001,In Transit,<NA>,False


In [465]:
df.tail()

,Shipment_ID,Shipment_Status,Shipment_Type,Shipment_Date,Shipment_Volume,Shipment_Event_ID,Ordinal_Shipment_ID,Return_Shipment_ID,Return_To_Sender_Flag,Tracking_ID,...,Claim_Amount,Currency_Code,Reporting_Date,Weight_kg,Length_cm,GPS_Jump,Duplicate_Stop,Skipped_Stop,Impossible_Order,Invalid_Return
483,SHP-100100,Picked Up,forward,15-Jul-2026,4290,SEV-000484,1,<NA>,0,TRK-9098,...,NaN,Rs,15 July 2026,16.14,15.0,False,True,False,False,False
484,SHP-100100,In Transit,Forward,"Jul 15, 2026",4290,SEV-000485,1,<NA>,n,TRK-9098,...,NaN,INR,16-Jul-2026,16.14,15.0,False,True,False,False,False
487,SHP-100101,Created,Forward,2026/09/04,7018,SEV-000488,1,<NA>,FALSE,TRK-9099,...,NaN,Rs,"Sep 04, 2026",5.05,29.0,False,True,False,False,False
488,SHP-100101,Picked Up,Forward,2026/09/04,7018,SEV-000489,1,<NA>,no,TRK-9099,...,NaN,inr,"Sep 04, 2026",5.05,29.0,False,True,False,False,False
489,SHP-100101,In Transit,Forward,2026-09-04,7018,SEV-000490,1,<NA>,FALSE,TRK-9099,...,NaN,INR,"Sep 05, 2026",5.05,29.0,False,True,False,False,False


# 19. Mask customer names, phone numbers, and street-address details in analytics extracts.

In [466]:
df.columns[df.columns.str.contains('name|phone|address', case=False)].tolist()

['Carrier_Name',
 'Depot_Name',
 'Warehouse_Name',
 'Customer_Name',
 'Customer_Phone',
 'Street_Address',
 'Address_Line_1',
 'Address_Line_2',
 'Location_Name',
 'Recipient_Name']

customer names

In [467]:
df['Customer_Name'] = 'Mask'
df['Recipient_Name'] = 'Mask'


phone no

In [468]:
df['Customer_Phone'] = '********'

Address

In [469]:
df['Street_Address'] = 'Masked'
df['Address_Line_1'] = 'Masked'
df['Address_Line_2'] = 'Masked'

In [470]:
df.tail()

,Shipment_ID,Shipment_Status,Shipment_Type,Shipment_Date,Shipment_Volume,Shipment_Event_ID,Ordinal_Shipment_ID,Return_Shipment_ID,Return_To_Sender_Flag,Tracking_ID,...,Claim_Amount,Currency_Code,Reporting_Date,Weight_kg,Length_cm,GPS_Jump,Duplicate_Stop,Skipped_Stop,Impossible_Order,Invalid_Return
483,SHP-100100,Picked Up,forward,15-Jul-2026,4290,SEV-000484,1,<NA>,0,TRK-9098,...,NaN,Rs,15 July 2026,16.14,15.0,False,True,False,False,False
484,SHP-100100,In Transit,Forward,"Jul 15, 2026",4290,SEV-000485,1,<NA>,n,TRK-9098,...,NaN,INR,16-Jul-2026,16.14,15.0,False,True,False,False,False
487,SHP-100101,Created,Forward,2026/09/04,7018,SEV-000488,1,<NA>,FALSE,TRK-9099,...,NaN,Rs,"Sep 04, 2026",5.05,29.0,False,True,False,False,False
488,SHP-100101,Picked Up,Forward,2026/09/04,7018,SEV-000489,1,<NA>,no,TRK-9099,...,NaN,inr,"Sep 04, 2026",5.05,29.0,False,True,False,False,False
489,SHP-100101,In Transit,Forward,2026-09-04,7018,SEV-000490,1,<NA>,FALSE,TRK-9099,...,NaN,INR,"Sep 05, 2026",5.05,29.0,False,True,False,False,False


# 20. Flag duplicate tracking IDs across active shipments and unresolved location mappings.

Flag duplicate tracking IDs

In [471]:
df['Duplicate_Tracking_ID'] = df.duplicated(
    subset=['Tracking_ID'], keep=False
)

Flag missing location mappings

In [472]:
df['Missing_Location'] = df['Location_Name'].isna()

In [473]:
df[['Tracking_ID', 'Shipment_Status', 'Location_Name',
          'Duplicate_Tracking_ID', 'Missing_Location']].head()

,Tracking_ID,Shipment_Status,Location_Name,Duplicate_Tracking_ID,Missing_Location
4,TRK-9001,Delivered,Bhubaneswar Depot,True,False
3,TRK-9001,Out_For_Delivery,Bhubaneswar Depot,True,False
0,TRK-9001,Created,WH-NORTH,True,False
1,TRK-9001,Picked Up,WH-NORTH,True,False
2,TRK-9001,In Transit,Transit Hub,True,False


 Keep the time zone when saving to Excel

In [474]:

for col in df.columns:
    if isinstance(df[col].dtype, pd.DatetimeTZDtype):
        df[col] = df[col].astype(str)


df.to_excel("Logistics_Delivery_Cleaned.xlsx", index=False)

print("File saved with time zones!")

File saved with time zones!
